# AssureX Claim Decisioning — Multi-Model Classification & Hyperparameter Tuning

**Author:** AssureX Machine Learning Engineering Team  
**Target Variable:** `class_label` (`Valid Claim`, `Invalid Claim`, `Manual Review`)  
**Input Feature Matrices:** `train_features.csv` (N=1,050), `val_features.csv` (N=225), `test_features.csv` (N=225)  

This Google Colab / Jupyter notebook implements a complete competitive modeling and tuning pipeline:
1. **Environment Setup & Data Ingestion:** Loads `train_features.csv`, `val_features.csv`, and `test_features.csv`.
2. **Candidate Classifiers:**
   - **Random Forest (`RandomForestClassifier`)**
   - **XGBoost (`XGBClassifier`)**
   - **Support Vector Machine (`SVC` with `StandardScaler` pipeline)**
3. **Hyperparameter Tuning:** 5-fold Stratified Cross-Validation (`GridSearchCV`) optimizing `macro-F1`.
4. **Validation Benchmarking:** Evaluates Accuracy, Precision (Macro/Weighted), Recall (Macro/Weighted), F1 (Weighted), Macro-F1, and Confusion Matrices.
5. **Side-by-Side Model Comparison Table:** Direct performance comparison across all 3 models.
6. **Final Test Set Evaluation:** Evaluates winning model on `test_features.csv`, prints per-class recall, and audits **Manual Review Recall** with operational guardrails.
7. **Artifact Persistence:** Saves winning classifier to `model/claim_classifier.pkl` and preprocessing pipeline to `model/preprocessing.pkl`.

In [ ]:
# ==============================================================================
# Step 0: Dependency Installation & Imports
# ==============================================================================
!pip install -q xgboost scikit-learn joblib pandas numpy matplotlib seaborn

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
TARGET_CLASSES = ['Valid Claim', 'Invalid Claim', 'Manual Review']

print("All core dependencies imported successfully!")

---  
## Step 1: Feature Matrix Ingestion & Preparation

We load `train_features.csv`, `val_features.csv`, and `test_features.csv`. The feature matrix contains 22 predictors (continuous financial/age metrics, integer counts, binary evidentiary flags, frequency/target risk encodings, and one-hot category indicators) alongside the identifier `claim_id` and target `class_label`.

In [ ]:
def locate_file(filename: str) -> str:
    candidates = [
        filename,
        os.path.join("dataset", filename),
        os.path.join("/content", filename),
        os.path.join("/content", "dataset", filename),
        os.path.join("..", filename),
        os.path.join("..", "dataset", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Cannot find {filename} in {candidates}")

train_df = pd.read_csv(locate_file("train_features.csv"))
val_df = pd.read_csv(locate_file("val_features.csv"))
test_df = pd.read_csv(locate_file("test_features.csv"))

# Isolate predictors from identifiers and target
non_feature_cols = ['claim_id', 'class_label']
feature_cols = [c for c in train_df.columns if c not in non_feature_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df['class_label'].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df['class_label'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['class_label'].copy()

# Integer label encoder for XGBoost compatibility
label_encoder = LabelEncoder()
label_encoder.fit(TARGET_CLASSES)
y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print(f"[+] Feature count: {len(feature_cols)}")
print(f"[+] Train split:   {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"[+] Val split:     {X_val.shape[0]} samples, {X_val.shape[1]} features")
print(f"[+] Test split:    {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"[+] Label mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

---  
## Step 2: Evaluation Framework & Metric Helpers

We configure Stratified 5-Fold Cross Validation (`StratifiedKFold`) optimizing `f1_macro`. Macro-F1 treats all three decision classes equally, ensuring that borderline **Manual Review** cases are not penalized relative to high-volume standard approvals/rejections.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model_performance(y_true, y_pred, model_name: str) -> dict:
    """Calculates standardized classification metrics and confusion matrix."""
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    prec_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    rec_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=TARGET_CLASSES)
    report = classification_report(y_true, y_pred, labels=TARGET_CLASSES, output_dict=True, zero_division=0)

    return {
        "model_name": model_name,
        "accuracy": acc,
        "precision_macro": prec_macro,
        "precision_weighted": prec_weighted,
        "recall_macro": rec_macro,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
        "f1_macro": f1_macro,
        "confusion_matrix": cm,
        "report": report
    }

trained_candidates = {}
val_evaluation_records = []
print("Evaluation framework configured.")

---  
## Step 3: Candidate Model 1 — Random Forest Classifier

Random Forest builds an ensemble of decorrelated decision trees. It is inherently non-linear, handles mixed scale numerical features, and resists overfitting.

In [ ]:
print("[*] Tuning Random Forest with 5-Fold Stratified Cross-Validation...")

rf_base = RandomForestClassifier(random_state=SEED)
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 12, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced', None]
}

grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

print(f"[+] Random Forest Best 5-Fold CV Macro-F1: {grid_rf.best_score_:.4f}")
print(f"[+] Optimal Hyperparameters: {grid_rf.best_params_}")

# Evaluate on Validation Set
rf_val_preds = best_rf.predict(X_val)
rf_eval = evaluate_model_performance(y_val, rf_val_preds, "Random Forest")
rf_eval['cv_macro_f1'] = grid_rf.best_score_
val_evaluation_records.append(rf_eval)

trained_candidates['Random Forest'] = {
    'model': best_rf,
    'cv_score': grid_rf.best_score_,
    'params': grid_rf.best_params_,
    'requires_encoding': False
}

print(f"[+] Random Forest Validation Accuracy: {rf_eval['accuracy']:.4f}, Val Macro-F1: {rf_eval['f1_macro']:.4f}")

---  
## Step 4: Candidate Model 2 — XGBoost Classifier

XGBoost employs second-order gradient boosting, tree pruning, and $L_1/L_2$ regularization. It is particularly effective on tabular datasets with non-linear threshold rules (e.g. date boundaries near expiry).

In [ ]:
print("[*] Tuning XGBoost with 5-Fold Stratified Cross-Validation...")

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=SEED
)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.03, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train_encoded)
best_xgb = grid_xgb.best_estimator_

print(f"[+] XGBoost Best 5-Fold CV Macro-F1: {grid_xgb.best_score_:.4f}")
print(f"[+] Optimal Hyperparameters: {grid_xgb.best_params_}")

# Evaluate on Validation Set
xgb_val_encoded_preds = best_xgb.predict(X_val)
xgb_val_preds = label_encoder.inverse_transform(xgb_val_encoded_preds)
xgb_eval = evaluate_model_performance(y_val, xgb_val_preds, "XGBoost")
xgb_eval['cv_macro_f1'] = grid_xgb.best_score_
val_evaluation_records.append(xgb_eval)

trained_candidates['XGBoost'] = {
    'model': best_xgb,
    'cv_score': grid_xgb.best_score_,
    'params': grid_xgb.best_params_,
    'requires_encoding': True
}

print(f"[+] XGBoost Validation Accuracy: {xgb_eval['accuracy']:.4f}, Val Macro-F1: {xgb_eval['f1_macro']:.4f}")

---  
## Step 5: Candidate Model 3 — Support Vector Machine (with Scaled Features)

Support Vector Machines construct optimal hyperplanes in a high-dimensional kernel space. Because SVMs rely on Euclidean distance calculations, **feature scaling via `StandardScaler` is strictly mandatory** to prevent continuous features like `purchase_price` or `product_age_days` from dominating binary indicators.

In [ ]:
print("[*] Tuning Support Vector Machine with StandardScaler Pipeline...")

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=SEED))
])

param_grid_svm = {
    'svm__C': [0.5, 1.0, 5.0, 10.0],
    'svm__gamma': ['scale', 'auto', 0.05],
    'svm__kernel': ['rbf'],
    'svm__class_weight': ['balanced', None]
}

grid_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_svm.fit(X_train, y_train)
best_svm = grid_svm.best_estimator_

print(f"[+] SVM Best 5-Fold CV Macro-F1: {grid_svm.best_score_:.4f}")
print(f"[+] Optimal Hyperparameters: {grid_svm.best_params_}")

# Evaluate on Validation Set
svm_val_preds = best_svm.predict(X_val)
svm_eval = evaluate_model_performance(y_val, svm_val_preds, "Support Vector Machine")
svm_eval['cv_macro_f1'] = grid_svm.best_score_
val_evaluation_records.append(svm_eval)

trained_candidates['Support Vector Machine'] = {
    'model': best_svm,
    'cv_score': grid_svm.best_score_,
    'params': grid_svm.best_params_,
    'requires_encoding': False
}

print(f"[+] SVM Validation Accuracy: {svm_eval['accuracy']:.4f}, Val Macro-F1: {svm_eval['f1_macro']:.4f}")

---  
## Step 6: Validation Set Comparison Table & Confusion Matrices

We tabulate all three classifiers side by side across 5-fold cross-validation score and validation performance metrics.

In [ ]:
comp_table_data = []
for ev in val_evaluation_records:
    comp_table_data.append({
        "Classifier": ev["model_name"],
        "5-Fold CV Macro-F1": f"{ev['cv_macro_f1']:.4f}",
        "Val Accuracy": f"{ev['accuracy']:.4f}",
        "Val Precision (Macro)": f"{ev['precision_macro']:.4f}",
        "Val Recall (Macro)": f"{ev['recall_macro']:.4f}",
        "Val F1 (Weighted)": f"{ev['f1_weighted']:.4f}",
        "Val Macro-F1": f"{ev['f1_macro']:.4f}",
    })

comparison_df = pd.DataFrame(comp_table_data)
print("\n" + "="*90)
print("MODEL VALIDATION BENCHMARK COMPARISON (val_features.csv, N=225)")
print("="*90)
display(comparison_df)

# Render side-by-side Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, ev in enumerate(val_evaluation_records):
    cm = ev['confusion_matrix']
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[idx],
        xticklabels=TARGET_CLASSES,
        yticklabels=TARGET_CLASSES,
        cbar=False
    )
    axes[idx].set_title(f"{ev['model_name']}\n(Val Macro-F1: {ev['f1_macro']:.4f})", fontsize=12, weight='bold')
    axes[idx].set_xlabel("Predicted Decision Class")
    axes[idx].set_ylabel("True Decision Class")

plt.tight_layout()
plt.show()

---  
## Step 7: Best Model Selection & Final Test Set Evaluation

We select the winning classifier based primarily on `Val Macro-F1` and secondarily on `Val Accuracy`. We then evaluate the champion model on the un-seen holdout `test_features.csv` dataset, inspect per-class recall, and **audit the Manual Review recall guardrail**.

In [ ]:
# Determine winning model
best_val_record = max(val_evaluation_records, key=lambda x: (x['f1_macro'], x['accuracy']))
winner_name = best_val_record['model_name']
winner_bundle = trained_candidates[winner_name]
champion_model = winner_bundle['model']

print(f"[+] WINNING MODEL: {winner_name}")
print(f"    Validation Macro-F1: {best_val_record['f1_macro']:.4f}")
print(f"    Validation Accuracy: {best_val_record['accuracy']:.4f}")

# Final Evaluation on Test Set
if winner_bundle['requires_encoding']:
    test_preds_raw = champion_model.predict(X_test)
    test_preds = label_encoder.inverse_transform(test_preds_raw)
else:
    test_preds = champion_model.predict(X_test)

test_eval = evaluate_model_performance(y_test, test_preds, winner_name)

print("\n" + "="*80)
print(f"CHAMPION MODEL TEST SET EVALUATION ({winner_name} on test_features.csv, N=225)")
print("="*80)
print(f"Test Accuracy:           {test_eval['accuracy']:.4f} ({test_eval['accuracy']*100:.2f}%)")
print(f"Test Precision (Macro):  {test_eval['precision_macro']:.4f}")
print(f"Test Recall (Macro):     {test_eval['recall_macro']:.4f}")
print(f"Test F1 (Weighted):      {test_eval['f1_weighted']:.4f}")
print(f"Test Macro-F1:           {test_eval['f1_macro']:.4f}")

print("\nPER-CLASS RECALL & PERFORMANCE METRICS:")
print("-"*65)
print(f"{'Decision Class':<18} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Support':<8}")
print("-"*65)

manual_review_recall = 0.0
for cls in TARGET_CLASSES:
    metrics = test_eval['report'][cls]
    p, r, f, s = metrics['precision'], metrics['recall'], metrics['f1-score'], int(metrics['support'])
    print(f"{cls:<18} | {p:<10.4f} | {r:<10.4f} | {f:<10.4f} | {s:<8}")
    if cls == 'Manual Review':
        manual_review_recall = r
print("-"*65)

# Manual Review Recall Audit
print("\n[OPERATIONAL GUARDRAIL AUDIT: MANUAL REVIEW RECALL]")
if manual_review_recall >= 0.85:
    print(f" [PASS - HEALTHY] Manual Review Recall is {manual_review_recall:.4f} (>= 0.85).")
    print("                 Borderline/ambiguous claims are safely and reliably routed to human adjusters.")
elif manual_review_recall >= 0.75:
    print(f" [WARNING - MODERATE] Manual Review Recall is {manual_review_recall:.4f} (< 0.85).")
    print("                     Threshold calibration recommended to minimize leakage of ambiguous claims.")
else:
    print(f" [CRITICAL ALERT - LOW RECALL] Manual Review Recall is {manual_review_recall:.4f} (< 0.75)!")
    print("                               High risk of auto-approving or auto-rejecting borderline claims.")

# Test Confusion Matrix Plot
plt.figure(figsize=(7, 5.5))
sns.heatmap(
    test_eval['confusion_matrix'],
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=TARGET_CLASSES,
    yticklabels=TARGET_CLASSES
)
plt.title(f"Test Set Confusion Matrix — {winner_name}\n(Test Macro-F1: {test_eval['f1_macro']:.4f})", fontsize=13, weight='bold', pad=12)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()

---  
## Step 8: Model & Preprocessing Artifact Persistence

We serialize the champion model and its full preprocessing pipeline bundle to `model/` using `joblib`:
- `model/claim_classifier.pkl`: Fitted winning classifier.
- `model/preprocessing.pkl`: Scaler, LabelEncoder, feature ordering, and metadata bundle.

In [ ]:
os.makedirs("model", exist_ok=True)
model_path = os.path.join("model", "claim_classifier.pkl")
prep_path = os.path.join("model", "preprocessing.pkl")

# Fit full scaler for inference
full_scaler = StandardScaler()
full_scaler.fit(X_train)

preprocessing_artifacts = {
    "feature_names": feature_cols,
    "scaler": full_scaler,
    "label_encoder": label_encoder,
    "target_classes": TARGET_CLASSES,
    "requires_encoding": winner_bundle['requires_encoding'],
    "model_name": winner_name,
    "test_metrics": {
        "accuracy": test_eval['accuracy'],
        "macro_f1": test_eval['f1_macro'],
        "manual_review_recall": manual_review_recall
    }
}

joblib.dump(champion_model, model_path)
joblib.dump(preprocessing_artifacts, prep_path)

print(f"[+] Champion model serialized:         {model_path}")
print(f"[+] Preprocessing bundle serialized:    {prep_path}")
print("\n[SUCCESS] Model training and evaluation pipeline fully completed!")